## Prawn CSV → YOLO

In [7]:
from pathlib import Path
import pandas as pd
import shutil

# =========================================================
# PRAWN DATASET PATHS
# =========================================================

SOURCE_DIR = Path(
    r"C:\Users\NANDINI\Desktop\AquaCare_Complete\prawn\shrimp.v9-shrimp-base.tensorflow"
)

OUTPUT_DIR = Path(
    r"C:\Users\NANDINI\Desktop\AquaCare_Complete\prawn\prawn_yolo"
)

# =========================================================
# PRAWN DISEASE CLASSES
# =========================================================

CLASS_MAP = {
    "Healthy": 0,
    "WSSV": 1,
    "Black_Gill": 2,
    "WSSV_Black_Gill": 3
}

# Create folders
for split in ["train", "valid", "test"]:

    (OUTPUT_DIR / split / "images").mkdir(
        parents=True,
        exist_ok=True
    )

    (OUTPUT_DIR / split / "labels").mkdir(
        parents=True,
        exist_ok=True
    )


# =========================================================
# IDENTIFY DISEASE FROM FILENAME
# =========================================================

def get_disease(filename):

    name = str(filename).upper()

    # IMPORTANT:
    # Check combined disease FIRST
    if "WSSV_BG" in name or "WSSV-BG" in name:
        return "WSSV_Black_Gill"

    if name.startswith("WSSV"):
        return "WSSV"

    if name.startswith("BG"):
        return "Black_Gill"

    if name.startswith("HEALTHY"):
        return "Healthy"

    return None


# =========================================================
# CONVERT EACH SPLIT
# =========================================================

for split in ["train", "valid", "test"]:

    print("\n======================================")
    print("Processing:", split)
    print("======================================")

    csv_path = SOURCE_DIR / split / "_annotations.csv"

    if not csv_path.exists():
        print("❌ CSV not found:", csv_path)
        continue

    df = pd.read_csv(csv_path)

    required = [
        "filename",
        "width",
        "height",
        "class",
        "xmin",
        "ymin",
        "xmax",
        "ymax"
    ]

    missing = [
        col for col in required
        if col not in df.columns
    ]

    if missing:
        raise ValueError(
            f"Missing columns: {missing}"
        )

    print("Total annotations:", len(df))

    unknown_files = []

    # Group boxes belonging to same image
    for filename, group in df.groupby("filename"):

        disease = get_disease(filename)

        if disease is None:
            unknown_files.append(filename)
            continue

        class_id = CLASS_MAP[disease]

        # Find image
        source_image = SOURCE_DIR / split / filename

        if not source_image.exists():

            # Try searching by stem/name
            matches = list(
                (SOURCE_DIR / split).glob(
                    Path(filename).stem + "*"
                )
            )

            if len(matches) == 0:
                print("⚠️ Image missing:", filename)
                continue

            source_image = matches[0]

        # Copy image
        destination_image = (
            OUTPUT_DIR
            / split
            / "images"
            / source_image.name
        )

        shutil.copy2(
            source_image,
            destination_image
        )

        width = float(group.iloc[0]["width"])
        height = float(group.iloc[0]["height"])

        label_lines = []

        # Convert every bounding box
        for _, row in group.iterrows():

            xmin = float(row["xmin"])
            ymin = float(row["ymin"])
            xmax = float(row["xmax"])
            ymax = float(row["ymax"])

            # Pascal VOC → YOLO
            x_center = ((xmin + xmax) / 2) / width
            y_center = ((ymin + ymax) / 2) / height

            box_width = (xmax - xmin) / width
            box_height = (ymax - ymin) / height

            label_lines.append(
                f"{class_id} "
                f"{x_center:.6f} "
                f"{y_center:.6f} "
                f"{box_width:.6f} "
                f"{box_height:.6f}"
            )

        label_path = (
            OUTPUT_DIR
            / split
            / "labels"
            / f"{Path(filename).stem}.txt"
        )

        label_path.write_text(
            "\n".join(label_lines),
            encoding="utf-8"
        )

    print("✅ Images:",
          len(list((OUTPUT_DIR / split / "images").glob("*"))))

    print("✅ Labels:",
          len(list((OUTPUT_DIR / split / "labels").glob("*.txt"))))

    if unknown_files:
        print("\n⚠️ Unknown filename patterns:", len(unknown_files))
        for f in unknown_files[:10]:
            print("   ", f)


# =========================================================
# CREATE data.yaml
# =========================================================

yaml_text = """path: C:/Users/NANDINI/Desktop/AquaCare_Complete/prawn/prawn_yolo

train: train/images
val: valid/images
test: test/images

nc: 4

names:
  0: Healthy
  1: WSSV
  2: Black_Gill
  3: WSSV_Black_Gill
"""

yaml_path = OUTPUT_DIR / "data.yaml"

yaml_path.write_text(
    yaml_text,
    encoding="utf-8"
)

print("\n======================================")
print("✅ PRAWN YOLO DATASET READY")
print("======================================")

print("Output:", OUTPUT_DIR)
print("YAML  :", yaml_path)


Processing: train
Total annotations: 2311
✅ Images: 1611
✅ Labels: 1611

⚠️ Unknown filename patterns: 700
    IMG-20240414-WA0014_jpg.rf.bef9023feb35fa6104d18229fdeea4c4.jpg
    IMG-20240414-WA0015_jpg.rf.c4bfcefd03f4ce7220e46a6ccbedba3b.jpg
    IMG-20240414-WA0018_jpg.rf.8bfcacaa5ebf722eb61bcf22e4b96425.jpg
    IMG-20240414-WA0022_jpg.rf.26f0b578179fab0fbde026ecac7a01d8.jpg
    IMG-20240414-WA0030_jpg.rf.c63fa16dd66b1c8158135e5b8e6e8231.jpg
    IMG-20240414-WA0034_jpg.rf.ba9c8ab4e27af8d4a3ab5fa11501d910.jpg
    IMG-20240414-WA0036_jpg.rf.0738884aded42cd6b03481a7b194a0c3.jpg
    IMG-20240414-WA0037_jpg.rf.3a77b5bc2131e8109b0a251082406ab1.jpg
    IMG-20240414-WA0045_jpg.rf.afba0438bccd0b831789aa39524a6c4e.jpg
    IMG-20240414-WA0047_jpg.rf.ec6e833894ea5eada5e8e173247b68ec.jpg

Processing: valid
Total annotations: 658
✅ Images: 457
✅ Labels: 457

⚠️ Unknown filename patterns: 201
    IMG-20240414-WA0046_jpg.rf.e432d2b2cf3d3c9f37d213c89f95a05a.jpg
    IMG-20240414-WA0099_jpg.rf.c3512285

In [8]:
for split in ["train", "valid", "test"]:

    images = list(
        (OUTPUT_DIR / split / "images").glob("*")
    )

    labels = list(
        (OUTPUT_DIR / split / "labels").glob("*.txt")
    )

    print(
        f"{split}: images={len(images)}, labels={len(labels)}"
    )

print("\nClasses:")
for k, v in CLASS_MAP.items():
    print(v, "→", k)

train: images=1611, labels=1611
valid: images=457, labels=457
test: images=230, labels=230

Classes:
0 → Healthy
1 → WSSV
2 → Black_Gill
3 → WSSV_Black_Gill


## Fast Prawn training

In [9]:
from ultralytics import YOLO

prawn_model = YOLO("yolo11n.pt")

results = prawn_model.train(
    data=r"C:\Users\NANDINI\Desktop\AquaCare_Complete\prawn\prawn_yolo\data.yaml",
    epochs=5,
    imgsz=256,
    batch=2,
    workers=0,
    device="cpu",
    fraction=0.15,
    project=r"C:\Users\NANDINI\Desktop\AquaCare_Complete\prawn\models",
    name="prawn_fast_test",
    exist_ok=True
)

New https://pypi.org/project/ultralytics/8.4.172 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.171  Python-3.13.9 torch-2.14.1+cpu CPU (AMD Ryzen 5 5500U with Radeon Graphics)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=2, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\NANDINI\Desktop\AquaCare_Complete\prawn\prawn_yolo\data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=5, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=0.15, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=256, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=3

In [10]:
from pathlib import Path

best = Path(
    r"C:\Users\NANDINI\Desktop\AquaCare_Complete\prawn\models\prawn_fast_test\weights\best.pt"
)

print("Best model exists:", best.exists())
print(best)

Best model exists: True
C:\Users\NANDINI\Desktop\AquaCare_Complete\prawn\models\prawn_fast_test\weights\best.pt


In [11]:
from pathlib import Path

last = Path(
    r"C:\Users\NANDINI\Desktop\AquaCare_Complete\prawn\models\prawn_disease_final\weights\last.pt"
)

print("Last model exists:", last.exists())
print("Path:", last)

Last model exists: True
Path: C:\Users\NANDINI\Desktop\AquaCare_Complete\prawn\models\prawn_disease_final\weights\last.pt


In [1]:
from ultralytics import YOLO

prawn_model = YOLO("yolo11n.pt")

prawn_model.train(
    data=r"C:\Users\NANDINI\Desktop\AquaCare_Complete\prawn\prawn_yolo\data.yaml",
    epochs=30,
    imgsz=320,
    batch=2,
    workers=0,
    device="cpu",
    fraction=1.0,
    patience=8,
    project=r"C:\Users\NANDINI\Desktop\AquaCare_Complete\prawn\models",
    name="prawn_disease_final",
    exist_ok=True,
    plots=True
)

Ultralytics 8.4.171  Python-3.13.9 torch-2.14.1+cpu CPU (AMD Ryzen 5 5500U with Radeon Graphics)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=2, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\NANDINI\Desktop\AquaCare_Complete\prawn\prawn_yolo\data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=320, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=prawn_

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x0000022DE6475240>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
       

In [2]:
from ultralytics import YOLO

model = YOLO(
    r"C:\Users\NANDINI\Desktop\AquaCare_Complete\prawn\models\prawn_fast_test\weights\best.pt"
)

metrics = model.val(
    data=r"C:\Users\NANDINI\Desktop\AquaCare_Complete\prawn\prawn_yolo\data.yaml",
    split="test",
    plots=True
)

print("====================================")
print("🦐 PRAWN MODEL TEST RESULTS")
print("====================================")
print("mAP50    :", round(metrics.box.map50, 4))
print("mAP50-95 :", round(metrics.box.map, 4))

Ultralytics 8.4.171  Python-3.13.9 torch-2.14.1+cpu CPU (AMD Ryzen 5 5500U with Radeon Graphics)
YOLO11n summary (fused): 100 layers, 2,582,932 parameters, 0 gradients, 6.4 GFLOPs
WARNING val: Slow image access detected (ping: 0.20.0 ms, read: 12.26.3 MB/s, size: 22.6 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips
val: Scanning C:\Users\NANDINI\Desktop\AquaCare_Complete\prawn\prawn_yolo\test\labels.cache... 230 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 230/230 38.6Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 15/15 1.9it/s 7.8s0.5s
                   all        230        230      0.528       0.25      0.258      0.187
               Healthy         80         80          1          0      0.355      0.214
                  WSSV         66         66          1          0      0.292      0.223
            Black_

## prawn model

In [3]:
from pathlib import Path

prawn_models = list(
    Path(r"C:\Users\NANDINI\Desktop\AquaCare_Complete\prawn\models").rglob("best.pt")
)

print("🦐 Prawn models:")
for p in prawn_models:
    print(p)

🦐 Prawn models:
C:\Users\NANDINI\Desktop\AquaCare_Complete\prawn\models\prawn_fast_test\weights\best.pt
C:\Users\NANDINI\Desktop\AquaCare_Complete\prawn\models\prawn_disease_final\weights\best.pt


In [4]:
results = model.predict(
    source=r"C:\Users\NANDINI\Desktop\AquaCare_Complete\prawn\prawn_yolo\test\images",
    conf=0.25,
    save=True,
    project=r"C:\Users\NANDINI\Desktop\AquaCare_Complete\prawn\models",
    name="prawn_test_predictions",
    exist_ok=True
)

print("✅ Prawn test prediction completed")


image 1/230 C:\Users\NANDINI\Desktop\AquaCare_Complete\prawn\prawn_yolo\test\images\BG-1-img-1_jpg.rf.1211405422581d61308e7d5d95de844f.jpg: 256x256 1 Black_Gill, 105.8ms
image 2/230 C:\Users\NANDINI\Desktop\AquaCare_Complete\prawn\prawn_yolo\test\images\BG-10-img-2_jpg.rf.7c239b49452a58b6ed20492853e17c8f.jpg: 256x256 1 Black_Gill, 47.4ms
image 3/230 C:\Users\NANDINI\Desktop\AquaCare_Complete\prawn\prawn_yolo\test\images\BG-11-img-3_jpg.rf.b3a6b25aaf48d88e2ade1548dc51dacc.jpg: 256x256 1 Black_Gill, 47.2ms
image 4/230 C:\Users\NANDINI\Desktop\AquaCare_Complete\prawn\prawn_yolo\test\images\BG-12-img-1_jpg.rf.402b329aadb1db490012900b96cd3b79.jpg: 256x256 1 Black_Gill, 55.4ms
image 5/230 C:\Users\NANDINI\Desktop\AquaCare_Complete\prawn\prawn_yolo\test\images\BG-13-img-3_jpg.rf.3dc52841966252ba8e383eb015729333.jpg: 256x256 1 Black_Gill, 46.8ms
image 6/230 C:\Users\NANDINI\Desktop\AquaCare_Complete\prawn\prawn_yolo\test\images\BG-15-img-1_jpg.rf.5d4e9ee5eba21f671b17aab3def38ece.jpg: 256x256 